In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E02_regresion_lineal"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesion EPE E2 - Regresion lineal para predecir

**Curso "Herramientas de Ciencias de Datos" - Modalidad EPE - UPC - Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E02_regresion_lineal/notebook/EPE_S2_regresion.ipynb)

> Enfoque EPE: se prioriza la **intuicion y la decision de negocio** sobre el
> formalismo. Se trabaja la **interpretacion** de los coeficientes, la **evaluacion**
> del modelo y una **verificacion de supuestos** a nivel EPE -heterocedasticidad
> (Breusch-Pagan / HC3), autocorrelacion (Durbin-Watson), normalidad de residuos
> (Jarque-Bera) y redundancia (VIF)- presentada como **idea + consecuencia de negocio**,
> no como derivacion formal. El diagnostico fino de influencia (distancia de Cook) y la
> maquinaria formal de correccion quedan fuera del alcance de esta sesion.

## Objetivos de aprendizaje
Al terminar la sesion, el participante es capaz de:
1. Construir una **regresion lineal simple** y leer la **recta** como una regla de negocio.
2. Interpretar los **coeficientes** y el **R2** en lenguaje de negocio.
3. Pasar a la **regresion multiple**, leer coeficientes **parciales** y reconocer la **idea de multicolinealidad**.
4. Incorporar **variables categoricas** (dummies) y leer su efecto como diferencia respecto a una base.
5. **Evaluar** el modelo con datos de **prueba** (RMSE y MAE) y evitar el **sobreajuste**.

## Mapa de la sesion
| # | Bloque | Datos |
|---|---|---|
| A | De la correlacion a la prediccion: la recta de regresion simple | Advertising (200 mercados) |
| B | Regresion multiple y la idea de multicolinealidad | Ames Housing (1 456 casas) |
| C | Variables categoricas (dummies) y como leer su efecto | Ames Housing |
| D | Evaluar con datos de prueba: RMSE y MAE; evitar el sobreajuste | Ambos |

**Materiales hermanos:** guia de laboratorio `laboratorio/GUIA_LABORATORIO_E02.docx`,
plantillas `plantillas/reporte_regresion_gerencial.docx` y `plantillas/regresion_excel_configurada.xlsx`,
ejercicios `evaluacion/drills.docx`, entregable evaluable `evaluacion/entregable.docx`,
fuentes de actualidad las fuentes de actualidad de la sesión. Lecturas: ISLR cap. 3;
Provost & Fawcett (2013) cap. 4.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "seaborn": "0.13.2",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerias de la sesion
import os, sys, io
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Estetica (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_INK, UPC_GRAY = "#E4002B", "#2D2D2D", "#9AA0A6"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

def rmse(y, yhat):
    return float(mean_squared_error(y, yhat) ** 0.5)

# TRAZABILIDAD NUMERICA: se reportan las versiones EN USO -- las que de verdad calculan las
# cifras de abajo--, no las que se pidieron en el pip (que en Colab solo surten efecto tras
# reiniciar el runtime). Son las del venv del curso, donde el material de referencia de la sesión certifico
# cada numero de la guia del docente.
VERSIONES_CERTIFICADAS = {"numpy": "2.5.1", "pandas": "2.3.3", "scipy": "1.16.3",
                          "matplotlib": "3.11.1", "scikit-learn": "1.6.1",
                          "statsmodels": "0.14.6"}
VERSIONES_EN_USO = {"numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
                    "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__}
_dif = [f"{k} {VERSIONES_EN_USO[k]} (certificada {v})"
        for k, v in VERSIONES_CERTIFICADAS.items() if VERSIONES_EN_USO[k] != v]

print("Librerias OK -", "Colab" if "google.colab" in sys.modules else "entorno local")
print("  en uso:", ", ".join(f"{k} {v}" for k, v in VERSIONES_EN_USO.items()))
print("  " + ("todas coinciden con las certificadas (la matriz de versiones certificada del curso)."
              if not _dif else "AVISO, difieren de las certificadas: " + "; ".join(_dif)))


🔎 **Antes de cargar los datos.** Las dos bases de la sesion son
**Advertising** (200 mercados, para la recta simple) y **Ames Housing** (1 456 casas, para
la regresion multiple). En negocio, la pregunta que abre cualquier modelo es siempre la
misma: *que resultado se quiere predecir y con que variables*. Aqui: **ventas** a partir de
la inversion publicitaria, y **precio de vivienda** a partir de sus caracteristicas.

In [ ]:
# Rutas robustas (nbconvert local y Colab) + CARGA ENDURECIDA de datos.
# Espeja data/descargar_datos.py: multi-mirror con fallback + checksum SHA256 de la copia
# versionada + verificación de esquema (columnas y n de filas), para que Colab reproduzca
# lo mismo que en local sin deriva silenciosa. El 1.er espejo de cada archivo es el repo del
# curso (bytes versionados, se exige checksum); los demás son mirrors abiertos de respaldo
# (verificación de esquema). Ambas bases pesan < 1 MB: la copia del repo es también la
# MUESTRA versionada de respaldo.
import urllib.request, hashlib

def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
DATA = os.getcwd() if "google.colab" in sys.modules else os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E02_resultados.xlsx")

# Espejos por archivo (se prueban en orden). El PRIMERO es la carpeta de datos que ya
# viaja junto al cuaderno en Drive (LEEME.md de la carpeta de Colab): se descarga por su
# file id, sin montar Drive ni pedir permisos. El repo del curso y los mirrors abiertos
# quedan como respaldo si esa carpeta se mueve o el enlace cambia.
def _drive(file_id):
    return f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

DRIVE_ID = {
    "Advertising.csv": "13U-waxsQUeuHIsYgpaybK3odHmWqCiH8",
    "AmesHousing_kaggle.csv": "10aAKeI3cDiwFQ679XgToJvVwvBI3zIOe",
}
_REPO = ("https://raw.githubusercontent.com/jonatanfigueroagil-creator/"
         "Herramientas-de-Ciencias-de-Datos/master/Sesiones_EPE/E02_regresion_lineal/data/")
MIRRORS = {
    "Advertising.csv": [
        _drive(DRIVE_ID["Advertising.csv"]),
        _REPO + "Advertising.csv",
        "https://www.statlearning.com/s/Advertising.csv",
        "https://raw.githubusercontent.com/nguyen-toan/ISLR/master/dataset/Advertising.csv",
    ],
    "AmesHousing_kaggle.csv": [
        _drive(DRIVE_ID["AmesHousing_kaggle.csv"]),
        _REPO + "AmesHousing_kaggle.csv",
    ],
}
# Checksum SHA256 de la copia versionada (byte-identidad; verificado 14/08/2026).
SHA256 = {
    "Advertising.csv": "5d88819733db0d929716e9c8fa87f20fe1ec12a9ba2feb43ed7e69e585ace3ad",
    "AmesHousing_kaggle.csv": "25273b2e062575f43bad23211f4f9d0df28502c95cd8cc84a4172bfe552b6e93",
}
ESQUEMAS = {  # (columnas clave esperadas, n de filas esperado)
    "Advertising.csv": (["TV", "radio", "newspaper", "sales"], 200),
    "AmesHousing_kaggle.csv": (
        ["Id", "SalePrice", "GrLivArea", "OverallQual", "GarageCars",
         "TotalBsmtSF", "YearBuilt", "Neighborhood"], 1460),
}

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _verificar(df, nombre):
    cols, nrows = ESQUEMAS[nombre]
    faltan = [c for c in cols if c not in df.columns]
    if faltan:
        raise ValueError(f"[{nombre}] faltan columnas {faltan}; hay {list(df.columns)[:8]}...")
    if len(df) != nrows:
        raise ValueError(f"[{nombre}] se esperaban {nrows} filas y hay {len(df)}")

def asegurar(nombre):
    """Trae 'nombre' a DATA verificado (checksum del versionado / esquema). Idempotente."""
    ruta = os.path.join(DATA, nombre)
    if os.path.exists(ruta):  # idempotencia: no re-descarga si el local es valido
        try:
            if _sha256(ruta) == SHA256[nombre]:
                return ruta                       # byte-identico a la copia versionada
            _verificar(pd.read_csv(ruta, low_memory=False), nombre)
            print(f"  ~ {nombre} local con checksum distinto al versionado pero esquema OK; se usa.")
            return ruta
        except Exception as e:  # noqa: BLE001
            print(f"  ! {nombre} local invalido ({e}); se re-obtiene.")
    ultimo = None
    for url in MIRRORS[nombre]:
        try:
            print(f"Descargando {nombre} de {url.split('/')[2]}...")
            raw = urllib.request.urlopen(url, timeout=90).read()
            df = pd.read_csv(io.BytesIO(raw), low_memory=False)
            _verificar(df, nombre)
            if url.startswith(_REPO) and hashlib.sha256(raw).hexdigest() != SHA256[nombre]:
                raise ValueError("checksum del repo no coincide")
            with open(ruta, "wb") as f:
                f.write(raw)
            print(f"  + verificado y guardado -> {os.path.basename(ruta)}")
            return ruta
        except Exception as e:  # noqa: BLE001
            ultimo = e
            print(f"  ! mirror fallo ({e!r})")
    if nombre == "AmesHousing_kaggle.csv":  # fallback final: OpenML espeja el train de Kaggle
        print("  .. sin mirror; se carga via fetch_openml(42165)...")
        from sklearn.datasets import fetch_openml
        df = fetch_openml(data_id=42165, as_frame=True).frame
        _verificar(df, nombre)
        df.to_csv(ruta, index=False, encoding="utf-8")
        print(f"  + cargado via fetch_openml -> {os.path.basename(ruta)}")
        return ruta
    raise RuntimeError(f"No se pudo obtener {nombre} de ningun espejo. Ultimo: {ultimo!r}")

adv = pd.read_csv(asegurar("Advertising.csv"))
ames = pd.read_csv(asegurar("AmesHousing_kaggle.csv"), low_memory=False)
print("Advertising :", adv.shape, "->", ["TV", "radio", "newspaper", "sales"])
print("Ames Housing:", ames.shape)

---
## Parte A — De la correlación a la predicción: la recta de regresión

**Caso de negocio (Advertising).** Un anunciante invierte en tres canales (TV, radio,
prensa) en 200 mercados y registra las **ventas** (en miles de unidades) y el
**presupuesto** de cada canal (en miles de USD). La correlación de la sesión anterior
dice *si* dos variables se mueven juntas; la **regresión** da un paso más: cuantifica
**cuánto** cambia la venta por cada unidad invertida y permite **predecir**.

El modelo simple ajusta una **recta**: `ventas = b0 + b1 * TV`. `b1` (la **pendiente**)
es el número accionable; `b0` (el **intercepto**) es solo el punto de partida de la recta.

**❓ Qué se quiere averiguar.** ¿Cuántas unidades más se venden por cada mil dólares adicionales invertidos en televisión? La correlación ya dijo que TV y ventas se mueven juntas; lo que falta es el *cuánto*, en una cifra que quepa en un presupuesto.

- **Qué decide:** con ese número —y solo con él— se responde si conviene mover dinero hacia TV y qué retorno esperar por cada mil USD.
- **Antes de mirar el resultado:** si el **intervalo de confianza de la pendiente incluyera el 0**, ni siquiera se podría afirmar que invertir en TV aumenta las ventas. Si lo **excluye**, queda la segunda mitad de la pregunta —si el efecto es lo bastante grande para pagar la inversión—, que ninguna prueba contesta. Conviene mirar además el R²: una recta puede tener pendiente nítida y explicar poco.

🔎 **La celda clave del bloque.** El siguiente gráfico convierte la **nube de puntos** en una
**recta**: el salto de "hay relación" (correlación) a "predigo un número" (regresión).
Observar cómo la pendiente de la recta roja resume, en un solo número, cuánto suben las ventas
por cada mil USD invertidos en TV.

In [ ]:
# Recta de regresion simple ventas ~ TV (grafico ilustrativo, trazado de los datos).
b1, b0 = np.polyfit(adv["TV"], adv["sales"], 1)   # pendiente, intercepto
xx = np.linspace(adv["TV"].min(), adv["TV"].max(), 100)

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.scatter(adv["TV"], adv["sales"], s=22, color=UPC_GRAY, alpha=0.8, label="mercados")
ax.plot(xx, b0 + b1 * xx, color=UPC_RED, linewidth=2.4,
        label=f"recta ajustada: ventas = {b0:.2f} + {b1:.4f}*TV")
ax.set_xlabel("Presupuesto de TV (miles de USD)")
ax.set_ylabel("Ventas (miles de unidades)")
ax.set_title("De la nube de puntos a la recta de prediccion")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_advertising_recta.png"), bbox_inches="tight")
plt.show()
print(f"correlacion TV-ventas = {adv['TV'].corr(adv['sales']):.4f}")

In [ ]:
# Ajuste OLS con statsmodels: la tabla de coeficientes.
X = sm.add_constant(adv[["TV"]]); y = adv["sales"]
ols_tv = sm.OLS(y, X).fit()
ic = ols_tv.conf_int().loc["TV"]
print(f"Intercepto b0 = {ols_tv.params['const']:.4f}")
print(f"Pendiente  b1 = {ols_tv.params['TV']:.5f}  (p = {ols_tv.pvalues['TV']:.2e})")
print(f"IC 95% de b1  = [{ic[0]:.5f} ; {ic[1]:.5f}]")
print(f"R2            = {ols_tv.rsquared:.4f}")

# Comparacion rapida: y si la palanca fuera la radio en vez de la TV?
ols_radio = sm.OLS(y, sm.add_constant(adv[["radio"]])).fit()
print(f"\nComparacion canal radio: b1 = {ols_radio.params['radio']:.5f}  R2 = {ols_radio.rsquared:.4f}")

📖 **Dos detalles de implementación que conviene nombrar.**
- **`sm.add_constant(X)`** agrega una columna de unos al conjunto de predictores. Esa columna
  es la que permite a `statsmodels` estimar el **intercepto** `b0` (el ancla de la recta); sin
  ella el modelo pasaría forzado por el origen. Es un requisito de la librería, no un paso de
  negocio.
- **`random_state=42`** (más abajo, en la partición train/test) **fija la semilla** de la partición aleatoria
  que separa el 80 % de entrenamiento del 20 % de prueba. Fijarla hace el resultado
  **reproducible**: cualquiera que ejecute el cuaderno obtiene la misma partición y el mismo
  RMSE/MAE. Otra semilla daría una partición algo distinta: las métricas de prueba son de **esa**
  partición (un valor ilustrativo), no una constante universal.

📖 **Lectura de negocio (coeficientes y R2).**
- **Pendiente b1 = 0.0475.** Por cada **1 000 USD adicionales** invertidos en TV se
  asocian **~47.5 unidades más vendidas** (0.0475 miles de unidades). Es la regla
  accionable del modelo, y su **intervalo de confianza al 95 %** [0.042 ; 0.053] **no
  incluye el 0**: el efecto es **distinguible de cero** (no una casualidad de la muestra,
  p < 0.001) — aunque distinguible ≠ grande ≠ causal.
- **Intercepto b0 = 7.03.** Ventas esperadas con inversión de TV = 0; solo es el
  **ancla** de la recta, no se sobreinterpreta como una predicción de negocio.
- **R2 = 0.61.** La TV explica el **61 %** de la variación de las ventas; el resto lo
  explican otros factores (radio, prensa, estacionalidad). Un R2 alto **no** prueba
  causalidad ni garantiza buenas predicciones nuevas: por eso se evalúa fuera de muestra.
- **La radio** tiene una pendiente mayor (0.20) pero un R2 menor (0.33): mueve más por
  cada mil USD, pero explica menos de la venta total. Comparar palancas exige mirar
  **ambas** cifras, no una sola.

> ⚠️ **Riesgo - extrapolar fuera del rango.** La recta solo es fiable **dentro del rango
> observado** de inversión (los presupuestos de TV de estos 200 mercados). Usarla para
> predecir con una inversión mucho mayor que la máxima vista es **extrapolar**: el modelo
> nunca aprendió esa zona y puede fallar sin ninguna señal de advertencia.
>
> ⚠️ **Riesgo - un R2 alto no es suficiente.** El R2 mide **cuánto explica** el modelo en la
> muestra de ajuste; **no** prueba **causalidad** (que la TV *cause* las ventas) ni garantiza
> **buen ajuste predictivo** con datos nuevos. La causa se comprueba con diseño; la
> predicción, con el error fuera de muestra (Parte D).

**❓ Qué se quiere averiguar.** ¿El intervalo que se acaba de leer describe los datos, o descansa sobre el supuesto de varianza constante que la nube ventas~TV visiblemente incumple?

- **Qué decide:** si la cifra insignia del bloque se comunica como firme o con reservas. Un intervalo sostenido por un supuesto falso promete una precisión que el modelo no tiene.
- **Antes de mirar el resultado:** si el intervalo **bootstrap** cae **casi encima** del clásico, la conclusión no dependía del supuesto y la cifra es firme por dos rutas independientes. Si resultara **bastante más ancho**, el embudo de la nube habría inflado la confianza del intervalo clásico y el que se reporta pasaría a ser el bootstrap.

🎯 **¿Cuán firme es la cifra insignia?** El intervalo [0.042 ; 0.053] de la pendiente de TV es **clásico**: supone varianza constante de los errores. Como la nube ventas~TV es un **embudo** (los mercados con más inversión se dispersan más), conviene confirmar la cifra por una vía que **no** dependa de ese supuesto: el **bootstrap**. Se re-muestrean los 200 mercados **con reemplazo** miles de veces, se reajusta la recta en cada re-muestra y se mira la **dispersión empírica** de la pendiente. Si el intervalo bootstrap coincide con el clásico, la cifra insignia del curso es firme.

In [ ]:
# IC bootstrap de la cifra insignia (pendiente de TV): via INDEPENDIENTE del IC clasico.
# Bootstrap de PARES: se re-muestrean los 200 mercados con reemplazo y se reajusta la recta
# muchas veces; el IC sale de la dispersion empirica de la pendiente (sin suponer varianza
# constante). Semilla fija -> IC reproducible; misma conclusion que el IC clasico = cifra firme.
rng = np.random.default_rng(42)
B = 5000
TVv, Sv = adv["TV"].to_numpy(), adv["sales"].to_numpy()
Xa = np.column_stack([np.ones(len(adv)), TVv])
b1_boot = np.empty(B)
for i in range(B):
    idx = rng.integers(0, len(adv), len(adv))                    # re-muestra con reemplazo
    b1_boot[i] = np.linalg.lstsq(Xa[idx], Sv[idx], rcond=None)[0][1]
ic_b_low, ic_b_high = np.percentile(b1_boot, [2.5, 97.5])        # IC95 percentil

print(f"Pendiente b1 (OLS)             = {ols_tv.params['TV']:.5f}")
print(f"IC95 CLASICO   (statsmodels)   = [{ic[0]:.5f} ; {ic[1]:.5f}]  ancho {ic[1]-ic[0]:.5f}")
print(f"IC95 BOOTSTRAP (B={B}, seed 42) = [{ic_b_low:.5f} ; {ic_b_high:.5f}]  ancho {ic_b_high-ic_b_low:.5f}")
print(f"SE clasico = {ols_tv.bse['TV']:.5f}   |   SE bootstrap = {b1_boot.std(ddof=1):.5f}")

boot_tv_df = pd.DataFrame([{
    "cifra": "b1_TV (unidades por +1000 USD)",
    "punto": float(ols_tv.params["TV"]),
    "IC95_clasico_low": float(ic[0]), "IC95_clasico_high": float(ic[1]),
    "SE_clasico": float(ols_tv.bse["TV"]),
    "IC95_boot_low": float(ic_b_low), "IC95_boot_high": float(ic_b_high),
    "SE_boot": float(b1_boot.std(ddof=1)),
    "B": B, "semilla": 42, "metodo": "bootstrap de pares (percentil 2.5 / 97.5)"}])

📖 **Lectura (bootstrap de la cifra insignia).** El intervalo **bootstrap** cae casi encima del **clásico** [0.0422 ; 0.0528], apenas un poco más ancho. Traducido a negocio: cada **1 000 USD** adicionales en TV se asocian a entre **~42 y ~53 unidades** más vendidas, con 95 % de confianza — la misma conclusión por **dos rutas independientes** (fórmula clásica y re-muestreo empírico). La cifra insignia es firme; el intervalo clásico es un **piso** razonable de la incertidumbre: el bootstrap, que **no** asume varianza constante, lo ensancha solo levemente, señal de que el embudo de la nube ventas~TV no invalida la lectura.

**❓ Qué se quiere averiguar.** ¿En cuál de los tres canales conviene poner el siguiente millón? Y antes de eso: ¿existe realmente el efecto que una regresión simple le atribuye a la prensa?

- **Qué decide:** el reparto del presupuesto de medios. Un canal cuyo efecto propio se desvanece al controlar por los demás no merece una línea en el plan.
- **Antes de mirar el resultado:** si los coeficientes de la múltiple se parecen a los de las rectas simples, cada canal actuaba por su cuenta y las simples bastaban. Si alguno **cae hacia 0 y pierde significancia**, ese canal solo «vendía» porque se invierte a la vez que otro: variable de confusión típica, y el presupuesto que recibía estaba mal justificado.

🔎 **De dos rectas simples a una sola regresión múltiple.** Comparar
canales con dos regresiones **simples separadas** (una para TV, otra para radio) puede
**inducir a error**: cada recta ignora lo que hacen los demás canales y puede atribuir a uno el efecto
de otro (**confounding**, variable de confusión). La forma correcta de comparar palancas es una
**regresión múltiple** que las incluya a la vez y estime el efecto **parcial** de cada canal
*manteniendo los otros constantes*.

In [ ]:
# Regresion MULTIPLE de Advertising: los tres canales juntos (efecto parcial de cada uno).
Xadv = sm.add_constant(adv[["TV", "radio", "newspaper"]])
ols_adv_mult = sm.OLS(adv["sales"], Xadv).fit()
print(f"Multiple  sales ~ TV + radio + newspaper :  R2 = {ols_adv_mult.rsquared:.4f}")
for c in ["TV", "radio", "newspaper"]:
    print(f"    {c:10s} coef parcial = {ols_adv_mult.params[c]:+.5f}   p = {ols_adv_mult.pvalues[c]:.2e}")
print(f"\nContraste con las SIMPLES:  TV b1={ols_tv.params['TV']:.5f} (R2 {ols_tv.rsquared:.2f})"
      f"  |  radio b1={ols_radio.params['radio']:.5f} (R2 {ols_radio.rsquared:.2f})")
print("La prensa, sola, parece vender; en la multiple su coeficiente cae a ~0 (no significativo).")

📖 **Lectura (por qué la múltiple corrige a las simples).** Con los tres
canales juntos el R2 sube a **0.90** y el cuadro cambia:
- **TV** conserva su efecto (~+0.046 por mil USD) y **radio** el suyo (~+0.19): ambas son
  palancas reales.
- **Prensa (`newspaper`)** cae a un coeficiente **~0 y no significativo** (p ~ 0.86), pese a que
  una regresión simple de prensa daría una pendiente positiva. En la simple, la prensa "parecía"
  vender porque se **invierte junto con la radio**; al controlar por radio en la múltiple, su
  efecto propio se desvanece. Es el caso clásico de **variable de confusión**.

> ⚠️ **Riesgo - comparar palancas con regresiones simples separadas.** Una regresión
> simple por canal mezcla el efecto del canal con el de los demás con los que covaría. Para
> decidir "dónde invertir" se usa la **múltiple**, que aísla el efecto **parcial** de cada uno.

In [ ]:
# Evaluar fuera de muestra: particion train/test (80/20) y error en la moneda del negocio.
Xtr, Xte, ytr, yte = train_test_split(adv[["TV"]], adv["sales"], test_size=0.2, random_state=42)
lr_tv = LinearRegression().fit(Xtr, ytr)
pred = lr_tv.predict(Xte)
adv_rmse, adv_mae = rmse(yte, pred), float(mean_absolute_error(yte, pred))
adv_r2 = float(r2_score(yte, pred))
print(f"Pendiente (train) = {lr_tv.coef_[0]:.5f}  (global = {ols_tv.params['TV']:.5f})")
print(f"RMSE test = {adv_rmse:.3f} miles de unidades  |  MAE test = {adv_mae:.3f} miles de unidades")
print(f"R2 test   = {adv_r2:.4f}")

📖 **Lectura (evaluación fuera de muestra).** Sobre datos **no vistos**, el modelo se
equivoca en promedio **~2 440 unidades** (MAE) y **~3 190 unidades** con RMSE (que penaliza
más los errores grandes). La pendiente estimada solo con el 80 % de los datos (0.0465) es
casi idéntica a la global: el modelo es **estable** y no depende de mercados concretos. Un
modelo se compra por lo que **predecirá mañana** (error de prueba, en unidades del negocio),
no por lo que "explica" hoy.

> 💡 **Intuición.** "Estable" quiere decir que el modelo no cambia de historia según qué
> mercados entren en el entrenamiento. Un modelo que se sostiene con solo el 80 % de los datos
> es un modelo en el que se puede confiar para presupuestar.

---
## Parte B — Regresión múltiple y la idea de multicolinealidad

**Caso de negocio (Ames Housing).** El precio de una vivienda casi nunca depende de un
solo factor. La **regresión múltiple** combina varios predictores a la vez y estima el
**efecto parcial** de cada uno: cuánto aporta, *manteniendo los demás constantes*
(*ceteris paribus*). Se modela el precio de venta a partir de la superficie habitable
(`GrLivArea`), la calidad general (`OverallQual`), las plazas de garaje (`GarageCars`),
la superficie del sótano (`TotalBsmtSF`) y el año de construcción (`YearBuilt`).

Se retiran las **4 casas** con superficie > 4 000 pie2 (ventas atípicas que el autor del
dataset recomienda excluir) y se ajustan dos versiones: en **USD** (efecto en dólares por
unidad) y en **log del precio** (efecto en **porcentaje**, más sencillo de comunicar).

📖 **Por qué esta base existe, y por qué se filtra.** Ames no es un dataset de competencia: lo
construyó **Dean De Cock (2011)** para un objetivo docente declarado — hallar datos reales, actuales
y comprensibles para un lego, con variables y filas suficientes para que **ningún algoritmo
automático de selección resolviera solo** un proyecto de regresión. Descartó el Boston Housing por
sus precios de los años setenta y se negó a actualizarlos por inflación, ya que eso «cambiaría los
datos de reales a verosímiles»; pidió en cambio los registros de la **Oficina del Asesor Municipal
de Ames** —un caso real de valuación masiva de inmuebles— y del volcado de 113 variables conservó
las **80 que un comprador reconoce**. Entregó la base **sin depurar a propósito** y dejó una sola
regla al docente: apartar las viviendas de más de 4 000 pie². El criterio de fondo **no es el tamaño
sino el tipo de venta** —la mayoría son obra nueva todavía sin terminar al momento de la tasación,
cuyo precio no representa un valor de mercado— y el diagnóstico que él mismo prescribe es un gráfico
de precio contra superficie. De ahí la exclusión que aplica la celda de preparación: se filtra por
dominio, no por incomodidad.

> **Fuente:** De Cock, D. (2011), *Ames, Iowa: Alternative to the Boston Housing Data as an End of
> Semester Regression Project*, Journal of Statistics Education 19(3). En el archivo completo del
> autor (2 930 ventas) esas viviendas atípicas son **cinco**; en el subconjunto de Kaggle que usa
> esta sesión (1 460 filas), **cuatro**.

**❓ Qué se quiere averiguar.** ¿Cuánto vale, en porcentaje del precio, cada característica de una vivienda —un punto más de calidad, una plaza más de garaje, cien pies cuadrados más— una vez descontado el efecto de las demás?

- **Qué decide:** es la tabla que se lleva a la reunión. Con ella se tasa una casa, se prioriza una remodelación y se le explica a un cliente de dónde proviene el precio que se le ofrece.
- **Antes de mirar el resultado:** cada coeficiente debería traer el **signo que anticipa el sentido común** (más superficie, más precio) y una magnitud creíble; un signo invertido delata un problema de datos o de especificación, no un hallazgo. Conviene vigilar también cuánto sube el R² frente al modelo de un solo predictor: si apenas se moviera, los cuatro predictores añadidos no aportarían nada.

🔎 **La celda clave del bloque.** La siguiente celda ajusta el modelo **múltiple** y genera la
**tabla de coeficientes**: la salida que se lleva a la reunión. Cada coeficiente es un **efecto
parcial** (el aporte de una variable *manteniendo las demás constantes*), y el modelo en **log
del precio** los expresa en **porcentaje**, el lenguaje de la gerencia.

In [ ]:
# Preparacion y ajuste del modelo multiple (nivel USD y log del precio).
preds = ["GrLivArea", "OverallQual", "GarageCars", "TotalBsmtSF", "YearBuilt"]
d = ames[["SalePrice", "Neighborhood"] + preds].dropna().copy()
d = d[d["GrLivArea"] <= 4000].copy()
print(f"n = {len(d)} casas (tras excluir 4 atipicas > 4000 pie2 y filas incompletas)")

Xm = sm.add_constant(d[preds])
ols_nivel = sm.OLS(d["SalePrice"], Xm).fit()
ols_log = sm.OLS(np.log(d["SalePrice"]), Xm).fit()

print(f"\nModelo en USD : R2 = {ols_nivel.rsquared:.4f}  (F global p = {ols_nivel.f_pvalue:.1e})")
print("  coeficiente por unidad (USD):")
for p in preds:
    print(f"    {p:12s} {ols_nivel.params[p]:12,.2f}   p = {ols_nivel.pvalues[p]:.1e}")

print(f"\nModelo en log(precio): R2 = {ols_log.rsquared:.4f}  (efecto en % por unidad):")
for p in preds:
    ef = 100 * (np.exp(ols_log.params[p]) - 1)
    print(f"    {p:12s} {ef:+7.3f}%   p = {ols_log.pvalues[p]:.1e}")

# Contraste con el modelo simple (solo superficie).
ols_simple = sm.OLS(d["SalePrice"], sm.add_constant(d[["GrLivArea"]])).fit()
print(f"\nModelo SIMPLE (solo GrLivArea): R2 = {ols_simple.rsquared:.4f}  "
      f"vs multiple {ols_nivel.rsquared:.4f}")

🧮 **De log a porcentaje: la fórmula que produce cada titular.** El modelo en
**log del precio** estima coeficientes $\beta$ en la escala logarítmica. Para leerlos como
**cambio porcentual** del precio se aplica la **retro-transformación**:

$$\text{efecto \%} = 100\cdot\left(e^{\beta}-1\right)$$

Ejemplo con el barrio premium (Parte C): $\beta = 0.0405 \Rightarrow 100\cdot(e^{0.0405}-1) = +4.13\,\%$.
Para coeficientes pequeños $e^{\beta}-1 \approx \beta$; por eso `+0.028` en `GrLivArea` se lee como
**~+2.8 % por cada 100 pie2** (una **aproximación** del efecto exacto $100\cdot(e^{0.028}-1)=+2.84\,\%$).

📖 **Log vs. nivel: son dos modelos distintos.** Los titulares en **porcentaje** (+9.7 %
calidad, +7.2 % garaje, +4.1 % premium) provienen del modelo en **log del precio**. Las métricas
en **USD** (RMSE/MAE y el -43 % de la Parte D) provienen del modelo en **nivel (USD)**. Se citan
juntas por comodidad, pero son **dos ajustes**: uno comunica en %, el otro mide el error en
dólares. El modelo en log además **estabiliza la varianza** (ver el diagnóstico de
heterocedasticidad que sigue).

📖 **Lectura de negocio (coeficientes parciales).** Cada coeficiente se lee *manteniendo
lo demás constante*:
- **Calidad (`OverallQual`): +9.7 % de precio por cada punto** de calidad. Es la palanca
  más potente **por unidad natural** (por punto); **estandarizada** (+1 desviación típica,
  el movimiento comparable), la que predomina es la **superficie** (beta_std +0.356 vs +0.322
  de la calidad; ver los betas estandarizados de la Parte D).
- **Garaje (`GarageCars`): +7.2 %** por cada plaza adicional.
- **Superficie habitable (`GrLivArea`): +0.028 % por pie2**, es decir **~+2.8 % por cada
  100 pie2**. El sótano (`TotalBsmtSF`) suma **~+1.9 % por 100 pie2**.
- **Antigüedad (`YearBuilt`): +0.25 %** por cada año más nuevo.

Con el **log del precio**, cada coeficiente se interpreta como **cambio porcentual**, el
lenguaje natural de la gerencia. Y lo más importante: pasar de un solo predictor a cinco
eleva el R2 de **0.52 a 0.81** — la superficie sola explica la mitad del precio; sumar
calidad, garaje, sótano y año casi duplica el poder explicativo. Cada efecto es **parcial**:
el de la superficie **ya descuenta** calidad, garaje, sótano y antigüedad; no es la relación
bruta.

> ⚠️ **Riesgo - efecto parcial no es causa.** "+9.7 % de precio por punto de calidad" es una
> **asociación** dentro de este modelo (*manteniendo lo demás constante*), **no** la prueba de
> que subir la calidad *cause* ese aumento. Correlación, aun controlando por otras variables,
> **no es causalidad**: siempre puede faltar un factor no incluido (ubicación fina, estado del
> mercado) que mueva ambas variables a la vez.

In [ ]:
# La IDEA de multicolinealidad: predictores que miden "casi lo mismo".
corr_pred = d[preds].corr()
print("Correlacion entre predictores:")
print(corr_pred.round(2).to_string())

# Demostracion: agregar una variable REDUNDANTE desestabiliza el coeficiente.
# GarageCars (nº de plazas) y GarageArea (m2 de garaje) miden casi lo mismo.
g = ames[["SalePrice", "GrLivArea", "GarageCars", "GarageArea"]].dropna()
g = g[g["GrLivArea"] <= 4000]
corr_garaje = float(g["GarageCars"].corr(g["GarageArea"]))
m_solo = sm.OLS(np.log(g["SalePrice"]), sm.add_constant(g[["GrLivArea", "GarageCars"]])).fit()
m_ambas = sm.OLS(np.log(g["SalePrice"]),
                 sm.add_constant(g[["GrLivArea", "GarageCars", "GarageArea"]])).fit()
coef_solo = float(m_solo.params["GarageCars"])
coef_ambas = float(m_ambas.params["GarageCars"])
print(f"\ncorr(GarageCars, GarageArea) = {corr_garaje:.3f}  (muy alta: miden casi lo mismo)")
print(f"coef GarageCars  SOLO           = {coef_solo:.4f}")
print(f"coef GarageCars  + GarageArea   = {coef_ambas:.4f}  "
      f"(cae {100*(coef_solo-coef_ambas)/coef_solo:.0f} % al meter la variable redundante)")

📖 **Lectura (idea de multicolinealidad).** Cuando dos predictores estan **muy
correlacionados** entre si (aqui, plazas de garaje y area de garaje, r = 0.89) aportan
informacion **redundante**. El modelo no puede separar sus efectos: al incluir ambos, el
coeficiente de las plazas **cae de forma marcada** (de 0.23 a 0.16) y se vuelve **inestable** — cambia
de tamano segun que otras variables entren. La prediccion global apenas cambia, pero **la
interpretacion "cuanto aporta cada factor" deja de ser fiable**.

> ⚠️ **Riesgo - la multicolinealidad distorsiona la lectura de coeficientes.** Con variables
> redundantes el modelo **conserva su capacidad predictiva**, pero **el coeficiente de cada una por
> separado deja de ser confiable**: se vuelve inestable al entrar o salir la otra. Por eso no
> se leen como palancas independientes dos variables que miden lo mismo.

> **Idea, no formalismo.** El diagnostico formal de este problema usa un indicador llamado
> **VIF** y decide eliminar, combinar o transformar variables redundantes; excede el alcance
> de esta sesion EPE. Para el nivel EPE basta con la **intuicion**: si dos variables miden
> casi lo mismo, sobra una; conviene quedarse con la mas interpretable para el negocio.

---
## Parte C — Variables categóricas (dummies) y cómo leer su efecto

Muchos factores de negocio no son números sino **categorías**: la **zona** del inmueble, el
tipo de cliente, el canal. Para incluirlos en la regresión se convierten en **variables
indicadoras (dummies)**: una columna 0/1 por categoría. Aquí se marca si la vivienda está en
un **barrio premium** (NridgHt, NoRidge, StoneBr) y se lee su efecto **respecto al resto** de
barrios (la categoría de **referencia**).

In [ ]:
# Variable categorica: barrio premium (dummy 0/1) sobre el modelo en log(precio).
d["premium"] = d["Neighborhood"].isin(["NridgHt", "NoRidge", "StoneBr"]).astype(int)
print(f"Casas en barrio premium: {int(d['premium'].sum())} de {len(d)}")

Xc = sm.add_constant(d[preds + ["premium"]])
ols_cat = sm.OLS(np.log(d["SalePrice"]), Xc).fit()
efecto_premium = 100 * (np.exp(ols_cat.params["premium"]) - 1)
print(f"\nR2 sin premium = {ols_log.rsquared:.4f}   R2 con premium = {ols_cat.rsquared:.4f}")
print(f"coef premium = {ols_cat.params['premium']:.4f}  ->  efecto = {efecto_premium:+.2f}% "
      f"de precio (p = {ols_cat.pvalues['premium']:.3f})")
print("(el efecto se lee como DIFERENCIA respecto a un barrio NO premium, "
      "manteniendo lo demas constante)")

📖 **Lectura de negocio (dummies).** El coeficiente de `premium` es la **diferencia
respecto a la base** (un barrio no premium con la misma superficie, calidad, garaje, sótano y
año): estar en un barrio premium añade **~+4.1 % al precio** (p = 0.013, significativo). No se
lee como valor absoluto sino como **cuánto más caro** frente a la referencia. Regla práctica:
al crear dummies se **omite una categoría** (la base) para evitar la *colinealidad perfecta entre dummies* (*dummy variable trap*):
incluir todas más el intercepto genera columnas redundantes y el modelo no se puede estimar.

> 💡 **Intuición.** El coeficiente de una dummy nunca es un precio absoluto: es un **cuánto más
> (o menos)** frente a la categoría base. "+4.1 %" significa "4.1 % más caro que una casa
> equivalente en un barrio no premium", no "vale el 4.1 % del precio".

🔎 **Inferencia robusta (heterocedasticidad).** Los precios de vivienda rara
vez tienen **varianza constante** de los errores (los caros se dispersan más): es
**heterocedasticidad**. Cuando existe, los errores estándar **clásicos** (y con ellos los p e IC)
quedan mal calibrados. La siguiente celda (i) mide la heterocedasticidad con la prueba de
**Breusch-Pagan** y (ii) recalcula la inferencia con errores estándar **robustos HC3**. Los
coeficientes puntuales **no cambian**; cambia solo su incertidumbre.

In [ ]:
# Diagnostico de heterocedasticidad (Breusch-Pagan) + inferencia robusta HC3 (Ames).
from statsmodels.stats.diagnostic import het_breuschpagan

def _bp(modelo):
    lm, lm_p, F, F_p = het_breuschpagan(modelo.resid, modelo.model.exog)
    return lm, lm_p, F, F_p

bp_niv, bp_log, bp_cat = _bp(ols_nivel), _bp(ols_log), _bp(ols_cat)
print("Breusch-Pagan (H0: homocedasticidad):")
print(f"  Ames multiple NIVEL (USD)      LM = {bp_niv[0]:8.2f}   p = {bp_niv[1]:.2e}")
print(f"  Ames multiple LOG              LM = {bp_log[0]:8.2f}   p = {bp_log[1]:.2e}")
print(f"  Ames categorico LOG (+premium) LM = {bp_cat[0]:8.2f}   p = {bp_cat[1]:.2e}")

# Refit con covarianza robusta HC3 (mismos coeficientes; SE/t/p/IC robustos).
hc3_log = sm.OLS(np.log(d["SalePrice"]), Xm).fit(cov_type="HC3")
hc3_cat = sm.OLS(np.log(d["SalePrice"]), Xc).fit(cov_type="HC3")
ci_log = hc3_log.conf_int()

print("\nModelo LOG - clasico vs HC3 (efecto % = 100*(exp(coef)-1)):")
print(f"  {'predictor':12s} {'coef':>9s} {'SE_clas':>9s} {'SE_HC3':>9s} {'p_HC3':>10s}   IC95% HC3 en %")
filas_hc3 = []
for p in preds:
    coef = hc3_log.params[p]; lo, hi = ci_log.loc[p]
    efl, efh = 100 * (np.exp(lo) - 1), 100 * (np.exp(hi) - 1)
    print(f"  {p:12s} {coef:9.5f} {ols_log.bse[p]:9.5f} {hc3_log.bse[p]:9.5f} "
          f"{hc3_log.pvalues[p]:10.2e}   [{efl:+.3f}% ; {efh:+.3f}%]")
    filas_hc3.append({"predictor": p, "coef_log": coef,
                      "efecto_pct": 100 * (np.exp(coef) - 1),
                      "SE_clasico": float(ols_log.bse[p]), "SE_HC3": float(hc3_log.bse[p]),
                      "t_HC3": float(hc3_log.tvalues[p]), "p_clasico": float(ols_log.pvalues[p]),
                      "p_HC3": float(hc3_log.pvalues[p]),
                      "IC95_pct_low": efl, "IC95_pct_high": efh})

lo_pr, hi_pr = hc3_cat.conf_int().loc["premium"]
filas_hc3.append({"predictor": "premium", "coef_log": float(hc3_cat.params["premium"]),
                  "efecto_pct": 100 * (np.exp(hc3_cat.params["premium"]) - 1),
                  "SE_clasico": float(ols_cat.bse["premium"]), "SE_HC3": float(hc3_cat.bse["premium"]),
                  "t_HC3": float(hc3_cat.tvalues["premium"]), "p_clasico": float(ols_cat.pvalues["premium"]),
                  "p_HC3": float(hc3_cat.pvalues["premium"]),
                  "IC95_pct_low": 100 * (np.exp(lo_pr) - 1), "IC95_pct_high": 100 * (np.exp(hi_pr) - 1)})
print(f"\n  premium (categorico): coef {hc3_cat.params['premium']:.5f}  "
      f"SE {ols_cat.bse['premium']:.5f} -> HC3 {hc3_cat.bse['premium']:.5f}  "
      f"p_clasico {ols_cat.pvalues['premium']:.4f} -> p_HC3 {hc3_cat.pvalues['premium']:.4f}")

hc3_ames_df = pd.DataFrame(filas_hc3)
bp_ames_df = pd.DataFrame([
    {"modelo": "ames_multiple_nivel", "BP_LM": bp_niv[0], "BP_p": bp_niv[1], "BP_F": bp_niv[2], "BP_F_p": bp_niv[3]},
    {"modelo": "ames_multiple_log", "BP_LM": bp_log[0], "BP_p": bp_log[1], "BP_F": bp_log[2], "BP_F_p": bp_log[3]},
    {"modelo": "ames_categorico_log", "BP_LM": bp_cat[0], "BP_p": bp_cat[1], "BP_F": bp_cat[2], "BP_F_p": bp_cat[3]},
])

📖 **Lectura (que cambia con HC3).** La prueba de **Breusch-Pagan rechaza la
homocedasticidad** en los tres modelos (p << 0.05): hay heterocedasticidad. Detalle clave: el
modelo en **nivel (USD)** esta mucho mas afectado (LM ~ 174) que el modelo en **log** (LM ~ 56) —
**transformar a log es la mitigacion** de facto de la varianza creciente. Con errores **HC3**:
- Los **coeficientes no se mueven**: +9.7 % calidad y +7.2 % garaje siguen igual; cambian sus
  errores estandar (calidad 0.00475 -> 0.00518; garaje 0.00752 -> 0.00854) y, con ellos, los IC
  y p. Las cifras clasicas eran un **piso** de incertidumbre, no un techo.
- El **premium** es el caso interesante: su SE **baja** con HC3 (0.0162 -> 0.0145) y su p pasa de
  **0.013 a 0.005**, es decir se vuelve **mas** significativo, no menos. La significancia del
  barrio premium **resiste** la correccion robusta.

> **Idea, no formalismo.** El diagnostico y la correccion formal de la heterocedasticidad
> (Breusch-Pagan, White, errores HC0-HC3) exceden el alcance de esta sesion EPE. Para EPE
> basta la regla: **si el modelo es heterocedastico, se reportan errores
> robustos**, y el efecto puntual -el titular de negocio- no cambia; cambia el margen de
> confianza con que se comunica.

**❓ Qué se quiere averiguar.** ¿Con cuánta confianza se puede comunicar cada número de la tabla anterior? Los coeficientes se obtienen igual aunque el modelo incumpla sus supuestos: el software no emite ninguna advertencia.

- **Qué decide:** no si el modelo se aprueba o se reprueba, sino qué se promete y a quién. De aquí se deriva, por ejemplo, si la recomendación de compra se sostiene vivienda a vivienda o solo sobre una cartera.
- **Antes de mirar el resultado:** si la varianza de los errores **no es constante** (Breusch-Pagan), los efectos puntuales quedan pero los márgenes de confianza se recalculan con HC3. Si los residuos **no son normales**, con n = 1 456 la inferencia sobre los coeficientes se sostiene, aunque los intervalos por vivienda queden mal calibrados. Si algún **VIF pasara de 5**, dos predictores medirían casi lo mismo y sus coeficientes se volverían inestables.

🔎 **Tabla de supuestos: verificar antes de confiar en la lectura.** La lectura de un coeficiente OLS descansa sobre supuestos. La siguiente celda los **verifica** uno a uno sobre el modelo Ames en log (el que se interpreta) con una **prueba** cada uno —Breusch-Pagan (varianza), Durbin-Watson (autocorrelación), Jarque-Bera (normalidad de residuos) y VIF (redundancia)— y, lo importante, declara la **consecuencia de negocio** de cada resultado. No se trata de aprobar o reprobar el modelo, sino de saber **con cuánta confianza** se comunica cada número.

In [ ]:
# Tabla de supuestos VERIFICADA (Ames log): cada prueba con su veredicto y su consecuencia.
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor

resid = ols_log.resid
bp_lm, bp_p = bp_log[0], bp_log[1]                       # Breusch-Pagan (calculado arriba)
dw = durbin_watson(resid)                                # Durbin-Watson (autocorrelacion)
jb_stat, jb_p, skew, kurt = jarque_bera(resid)          # normalidad de residuos
vif = {p: variance_inflation_factor(Xm.values, i)       # VIF (Xm = const + 5 predictores)
       for i, p in enumerate(Xm.columns) if p != "const"}
vif_peor = max(vif, key=vif.get)

supuestos_df = pd.DataFrame([
    {"supuesto": "Homocedasticidad (varianza constante)", "prueba": "Breusch-Pagan",
     "estadistico": float(bp_lm), "p_o_valor": float(bp_p),
     "veredicto": "SE RECHAZA (heterocedastico)" if bp_p < 0.05 else "no se rechaza",
     "consecuencia": "IC/p clasicos mal calibrados -> se usa HC3 (ya hecho); HC3 puede ensanchar o estrechar el SE segun el caso (aqui +9 % calidad, +13 % garaje, -10 % premium)"},
    {"supuesto": "Errores sin autocorrelacion", "prueba": "Durbin-Watson",
     "estadistico": float(dw), "p_o_valor": np.nan,
     "veredicto": "OK (~2, sin autocorrelacion)" if 1.5 < dw < 2.5 else "revisar",
     "consecuencia": "dato transversal (no serie temporal): DW~2 es lo esperado; sin impacto"},
    {"supuesto": "Normalidad de los residuos", "prueba": "Jarque-Bera",
     "estadistico": float(jb_stat), "p_o_valor": float(jb_p),
     "veredicto": "SE RECHAZA (colas pesadas)" if jb_p < 0.05 else "no se rechaza",
     "consecuencia": "con n grande el TCL cubre la inferencia de COEFICIENTES; los INTERVALOS de "
                     "prediccion por casa quedan mal calibrados -> fiable en cartera, no casa a casa"},
    {"supuesto": "Sin multicolinealidad severa", "prueba": "VIF (peor: " + vif_peor + ")",
     "estadistico": float(vif[vif_peor]), "p_o_valor": np.nan,
     "veredicto": "OK (todos < 5)" if max(vif.values()) < 5 else "revisar",
     "consecuencia": "los 5 predictores no son redundantes entre si; la multicol se demostro "
                     "aparte con GarageArea (el VIF dispara al meter una variable que mide lo mismo)"},
])

print("Supuestos del modelo Ames en log (n =", len(d), "):")
print(supuestos_df[["supuesto", "prueba", "estadistico", "p_o_valor", "veredicto"]].to_string(index=False))
print("\nVIF por predictor (1 = sin redundancia; > 5 = alerta):")
for p, v in vif.items():
    print(f"    {p:12s} VIF = {v:5.2f}")
print(f"\nDurbin-Watson = {dw:.3f}   |   Jarque-Bera = {jb_stat:.1f} "
      f"(p = {jb_p:.1e}; asimetria {skew:.2f}, curtosis {kurt:.2f})")

📖 **Lectura (qué hacer con cada supuesto).** La tabla convierte cuatro pruebas en cuatro decisiones de negocio:
- **Heterocedasticidad — SE RECHAZA** (Breusch-Pagan LM ≈ 56, p ≈ 0). Ya está corregida con **HC3**: el efecto puntual no cambia, el margen de confianza sí. **HC3 re-mide** la incertidumbre bajo heterocedasticidad: puede **ensanchar** el SE (+9 % calidad, +13 % garaje) o **estrecharlo** (−10 % premium); garantiza **honestidad**, no conservadurismo.
- **Autocorrelación — OK** (Durbin-Watson ≈ 1.97, cerca de 2). Son casas, no una serie temporal: el supuesto no aplica de forma problemática y la prueba lo confirma.
- **Normalidad — SE RECHAZA** (Jarque-Bera muy elevado, colas pesadas). Con **n = 1 456** el Teorema del Límite Central cubre la inferencia sobre los **coeficientes**, pero los **intervalos de predicción por vivienda** quedan mal calibrados: por eso la recomendación de compra es fiable **en cartera** (agregado), no vivienda a vivienda — el efecto premium (~+4 %) es menor que el error típico por casa (MAE ~24 000 USD).
- **Multicolinealidad — OK** (VIF < 5 en los cinco predictores). No son redundantes entre sí; la inestabilidad se demostró **aparte**, al añadir `GarageArea` (que mide casi lo mismo que `GarageCars`). El VIF es el termómetro formal de esa idea.

> **Idea, no formalismo (EPE).** No es necesario retener cada prueba: basta la regla — *verificar antes de confiar*, y saber qué hacer si falla (log/HC3 para la varianza; leer en agregado si los residuos tienen colas; vigilar variables redundantes con el VIF).

📖 **Los argumentos del código robusto y de diagnóstico, en lenguaje sencillo.** Las celdas de arriba usan cuatro piezas de `statsmodels`/`numpy` cuyo *argumento* conviene nombrar, no solo el concepto:

- **`cov_type="HC3"`** (en el refit `sm.OLS(...).fit(cov_type="HC3")`). `cov_type` elige **cómo se calcula la matriz de varianzas-covarianzas de los coeficientes**, es decir, de dónde provienen los errores estándar. Por defecto es la clásica (supone varianza constante); `"HC3"` pide **errores estándar robustos a heterocedasticidad** de tipo *HC3* — la variante de los estimadores «sandwich» de White que **más penaliza las observaciones de alto apalancamiento**, la recomendada con muestras finitas. Los **coeficientes no cambian**; solo cambian sus SE, y con ellos los IC y los p.
- **`het_breuschpagan(resid, exog)`** (prueba de Breusch-Pagan). El **primer argumento** son los **residuos** del modelo (`modelo.resid`); el **segundo** es la **matriz de regresores** con la constante (`modelo.model.exog`). La prueba regresa los residuos al cuadrado contra esos regresores: si estos explican la varianza del error, hay heterocedasticidad. Devuelve cuatro valores — `LM, LM_p, F, F_p` (estadístico multiplicador de Lagrange y su p, más la versión F y su p).
- **`variance_inflation_factor(exog, i)`** (VIF de multicolinealidad). El **primer argumento** es la **matriz de predictores** (aquí `Xm.values`, con la constante) y el **segundo** es el **índice de la columna** cuyo VIF se calcula. Internamente regresa esa columna contra las demás y devuelve `1/(1−R²ⱼ)`: cuánto se **infla la varianza** de ese coeficiente por estar correlacionado con el resto (1 = sin redundancia; > 5 = alerta). Por eso se llama una vez **por columna** en un bucle.
- **`np.linalg.lstsq(A, b, rcond=None)`** (en el bootstrap de más arriba). `lstsq` resuelve el ajuste por mínimos cuadrados; **`rcond`** es el **umbral de valores singulares**: por debajo de `rcond·(mayor valor singular)` un valor singular se trata como cero al invertir (vía SVD/pseudoinversa), lo que estabiliza el cálculo si la matriz está mal condicionada. `rcond=None` **pide a NumPy su umbral por defecto moderno** (ligado a la precisión de la máquina) y, de paso, **silencia el `FutureWarning`** que aparecería al omitirlo.

> **Idea, no formalismo (EPE).** No es necesario retener estos argumentos: basta saber que `cov_type="HC3"` = «calcular errores robustos», que Breusch-Pagan y el VIF se alimentan de *residuos + regresores* y de *predictores + índice*, y que `rcond=None` solo fija el umbral numérico del ajuste. El titular de negocio no depende de ellos; sí la **calidad del margen de confianza** con que se comunica.

---

📖 **Los bloques no obvios del cuaderno, narrados (para el docente que prepara).** Además de los cuatro argumentos de arriba, hay decisiones de código que **mueven las cifras** y que no se ven a simple vista. Conviene tenerlas nombradas antes de clase, porque son las que explican por qué el número resulta exactamente así:

| Pieza de código | Dónde | Qué decide (y por qué la cifra depende de ella) |
|---|---|---|
| `dropna` sobre el subconjunto y `d[d["GrLivArea"] <= 4000]` | preparación de Ames | Fijan **n = 1 456 casas**. El `dropna` se aplica **solo a las columnas del modelo** (no a la base entera): filtrar antes o sobre más columnas cambiaría la muestra y con ella todos los coeficientes. El corte en 4 000 pie² es la exclusión que recomienda De Cock (2011), no un descarte ciego. |
| `random_state=42` en `train_test_split` | evaluación fuera de muestra | Fija **qué casas caen en prueba**. De ahí provienen RMSE 31 140, MAE 24 011 y el −43 %: son de **esa** partición. El validador confirma que la *conclusión* (múltiple < simple) se sostiene en cinco semillas, pero el porcentaje exacto oscila entre −35 % y −43 %. |
| `np.random.default_rng(42)` (bootstrap) | IC bootstrap de la pendiente | Semilla del generador **PCG64** de NumPy, distinta de `RandomState` (Mersenne Twister) que usa scikit-learn. Con la misma semilla el IC bootstrap se reproduce **bit a bit**; con otra, los extremos se mueven en el cuarto decimal. |
| `ddof=1` en `.std` | SE del bootstrap | Divide entre *n*−1 (varianza **muestral**), no entre *n*. Con `ddof=0` el SE saldría ligeramente menor: es la diferencia entre describir la muestra y estimar la población. |
| `low_memory=False` en `read_csv` | carga de Ames | Obliga a pandas a inferir el tipo de cada columna **con el archivo completo**. Sin él, en columnas mixtas puede inferir tipos distintos por bloques y colar un `object` donde debe haber número. |
| `data_only=True` (openpyxl, generador del deck) | figuras y slides | Lee el **valor calculado** de la celda, no su fórmula: garantiza que la slide muestre el mismo número que certificó el validador. |
| **No hay control de hilos BLAS** (`OMP_NUM_THREADS` y similares) | todo el cuaderno | Decisión consciente, y conviene saberla: NumPy resuelve el álgebra con una BLAS multihilo, y el **orden de sumación** puede variar según cuántos núcleos tenga la máquina. El efecto es del orden del épsilon de máquina (~10⁻¹⁶ relativo), muy por debajo de los decimales que se publican (0.0475, 31 140 USD), por lo que **no** se fija el número de hilos: hacerlo ralentizaría el cuaderno sin cambiar ninguna cifra de clase. Lo que sí protege las cifras son las **semillas** y las **versiones pineadas** (celda 1). Si alguna vez una cifra difiere en el último decimal entre dos máquinas, ésta es la causa, y no invalida la lectura. |

> **Cómo se comprueba todo esto.** el material de referencia de la sesión recomputa cada cifra desde los CSV crudos por una **vía independiente** (partición reproducida de forma manual, coeficientes por la ecuación normal, métricas promediadas de forma manual), la cruza contra el Excel y le exige identidades matemáticas y determinismo. Si una de estas decisiones de código cambiara, el validador dejaría de certificar.


---
## Parte D — Evaluar con datos de prueba: RMSE y MAE; evitar el sobreajuste

La prueba honesta de un modelo es su **error sobre datos que no vio al entrenar**. Se separa
la muestra en **entrenamiento** (80 %) y **prueba** (20 %), se ajusta con el primero y se mide
el error con el segundo, en la **moneda del negocio** (USD). Se comparan el modelo **simple**
(solo superficie) y el **múltiple** (cinco predictores).

**❓ Qué se quiere averiguar.** ¿De cuántos dólares se equivoca el modelo al tasar una casa que nunca vio, y compensa la complejidad del múltiple frente a la recta de un solo predictor?

- **Qué decide:** cuál de los dos modelos se pone a trabajar. Un modelo se elige por lo que acierta sobre datos nuevos, no por lo que explica sobre los que ya usó para ajustarse.
- **Antes de mirar el resultado:** si el error del múltiple resultara **parecido** al del simple, los cuatro predictores extra no aportan valor y conviene el modelo sencillo. Si resulta **claramente menor**, la complejidad se justifica. Y si el modelo tuviera un ajuste excelente en entrenamiento pero fallara en prueba, habría sobreajustado el ruido en vez de aprender la señal.

🔎 **La celda clave del bloque.** La siguiente celda mide el **error fuera de muestra** (RMSE y
MAE en USD) del modelo simple frente al múltiple. Es el **veredicto**: un modelo no se elige por
lo que explica hoy, sino por lo que acertará con casas que **no vio** al entrenar.

In [ ]:
# Comparacion train/test: modelo simple vs multiple (RMSE y MAE en USD).
Xtr, Xte, ytr, yte = train_test_split(d[preds], d["SalePrice"], test_size=0.2, random_state=42)
m_simple = LinearRegression().fit(Xtr[["GrLivArea"]], ytr)
m_mult = LinearRegression().fit(Xtr, ytr)

# La fila de Advertising va primero (otras unidades: miles de unidades), como referencia.
eval_rows = [{"modelo": "Advertising simple (ventas~TV)", "RMSE_test": adv_rmse,
              "MAE_test": adv_mae, "R2_test": adv_r2}]
for nombre, modelo, cols in [("Ames simple (GrLivArea)", m_simple, ["GrLivArea"]),
                             ("Ames multiple (5 predictores)", m_mult, preds)]:
    pr = modelo.predict(Xte[cols])
    eval_rows.append({"modelo": nombre, "RMSE_test": rmse(yte, pr),
                      "MAE_test": float(mean_absolute_error(yte, pr)),
                      "R2_test": float(r2_score(yte, pr))})
eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))
sim = eval_df[eval_df.modelo.str.startswith("Ames simple")].iloc[0]
mul = eval_df[eval_df.modelo.str.startswith("Ames multiple")].iloc[0]
print(f"\nEl modelo multiple reduce el RMSE de ~{sim.RMSE_test:,.0f} a ~{mul.RMSE_test:,.0f} USD "
      f"({100*(mul.RMSE_test-sim.RMSE_test)/sim.RMSE_test:.0f} %) y el MAE de "
      f"~{sim.MAE_test:,.0f} a ~{mul.MAE_test:,.0f} USD.")

📖 **Lectura de negocio (evaluar y evitar el sobreajuste).** Sobre datos no vistos, el
modelo **múltiple** yerra en promedio **~24 000 USD** (MAE) frente a los **~39 500 USD** del
modelo simple: sumar predictores **relevantes** baja el error de prueba **~43 %** (RMSE de
~54 500 a ~31 100 USD). Cuidado: agregar variables **siempre** mejora el ajuste *dentro* de la
muestra (el R2 nunca baja), pero solo mejora la **predicción nueva** si aportan señal real. El
**sobreajuste** aparece cuando el modelo se ajusta al ruido del entrenamiento: R2 alto en train,
error alto en test. Por eso la decisión se toma con el **error fuera de muestra**, en las
unidades del negocio, no con el R2 de entrenamiento.

> 💡 **Intuición - el sobreajuste.** Añadir variables **siempre** mejora el ajuste *dentro* de
> la muestra (el R2 de entrenamiento nunca baja), como un alumno que aprende de memoria el examen de
> práctica. La prueba honesta es un examen **nuevo**: el error sobre datos no vistos. Si el
> modelo rinde bien en train y falla en test, ha sobreajustado el ruido en lugar de aprender la señal.

In [ ]:
# Valor monetario del -43 %: traducir la mejora de error a USD (traza para la guia de negocio).
# El modelo multiple reduce el error de VALUACION por casa; sobre una cartera de N tasaciones ese
# ahorro se acumula. Es error de valuacion EVITADO (menor sobre/infra-valoracion esperada), NO
# utilidad: mide cuanto se estrecha la brecha entre precio estimado y precio real por casa.
sim = eval_df[eval_df.modelo.str.startswith("Ames simple")].iloc[0]
mul = eval_df[eval_df.modelo.str.startswith("Ames multiple")].iloc[0]
red_mae  = float(sim.MAE_test  - mul.MAE_test)          # USD/casa: menos error absoluto medio
red_rmse = float(sim.RMSE_test - mul.RMSE_test)         # USD/casa: menos error cuadratico medio
pct_rmse = float(100 * (mul.RMSE_test - sim.RMSE_test) / sim.RMSE_test)  # el titular: -43 %

print("Reduccion del error de valuacion por casa (simple -> multiple):")
print(f"  MAE : {sim.MAE_test:>10,.0f} -> {mul.MAE_test:>10,.0f} USD   ahorro {red_mae:>8,.0f} USD/casa")
print(f"  RMSE: {sim.RMSE_test:>10,.0f} -> {mul.RMSE_test:>10,.0f} USD   ahorro {red_rmse:>8,.0f} USD/casa ({pct_rmse:.0f} %)")

filas_valor = []
for N in [100, 500, 1000]:
    filas_valor.append({
        "tasaciones_N": N,
        "MAE_simple_USD_casa": float(sim.MAE_test),
        "MAE_multiple_USD_casa": float(mul.MAE_test),
        "ahorro_MAE_USD_casa": red_mae,
        "ahorro_RMSE_USD_casa": red_rmse,
        "reduccion_RMSE_pct": pct_rmse,
        "error_MAE_evitado_cartera_USD": red_mae * N,
        "error_RMSE_evitado_cartera_USD": red_rmse * N})
valor_error_df = pd.DataFrame(filas_valor)

print("\nError de valuacion evitado en cartera = ahorro por casa x N tasaciones:")
for r in filas_valor:
    print(f"  N = {r['tasaciones_N']:>4}   por MAE: {r['error_MAE_evitado_cartera_USD']:>12,.0f} USD"
          f"   |   por RMSE: {r['error_RMSE_evitado_cartera_USD']:>12,.0f} USD")


📖 **El −43 % en dinero (para la conversación con gerencia).** La mejora de modelo se comunica mejor en **USD que en porcentaje**. Pasar del modelo simple al múltiple **baja el error absoluto medio (MAE) de ~39 500 a ~24 000 USD por casa**: unos **~15 500 USD de error de valuación evitado por tasación**. Escalado a la operación, el ahorro se acumula: sobre **1 000 tasaciones** son **~15,5 millones de USD** de error de valuación esperado evitado (por MAE; ~23,4 millones si se mide con RMSE, que penaliza más los errores grandes).

> ⚠️ **Qué es y qué no es esta cifra.** Es **error de valuación evitado** —cuánto se estrecha, en promedio, la brecha entre el precio estimado y el real—, **no** utilidad ni ahorro de caja: un AVM que se equivoca menos reduce el riesgo de comprar por encima o vender por debajo del valor de mercado, pero traducir eso a beneficio exige el margen y la política de cada negocio. Además, el −43 % es de **una** partición train/test (varía con la semilla): la cifra en USD hereda esa incertidumbre y se lee como **orden de magnitud**, no como una constante.

**❓ Qué se quiere averiguar.** ¿Cuántas de las 292 tasaciones del conjunto de prueba conviene mandar a revisión manual, y qué palanca se le recomienda a un propietario que quiere subir el valor de su casa?

- **Qué decide:** un error medio no se opera; una lista de casas que alguien revisa, sí. Aquí la métrica se convierte en carga de trabajo del equipo y en consejo concreto al cliente.
- **Antes de mirar el resultado:** la regla que marca como candidata toda casa cuyo error supera **k veces el MAE** no tiene un k «correcto»: **cuanto más bajo el k, más larga la lista** y más caro el operativo, de modo que el umbral lo fija la capacidad real del equipo de revisión. Sobre las palancas, la respuesta depende del criterio de comparación: por unidad natural puede destacar una variable y por movimiento comparable —una desviación estándar— otra distinta. Conviene elegir el criterio antes de ver la tabla.

🔎 **De la métrica a la decisión operativa (traza completa).** Las celdas siguientes computan, sobre esta misma partición (semilla 42) y estos mismos modelos, las cifras que sostienen las decisiones de negocio de la guía: (1) el **recuento real de viviendas candidatas a revisión** con la regla `k·MAE`; (2) la **robustez del ahorro por casa** en cinco semillas de partición; (3) la **asimetría de los residuales en USD** (la que aplica al costo del error, no la del modelo en log); (4) los **coeficientes estandarizados** (qué palanca mueve más el precio por movimiento comparable); (5) las **palancas en USD sobre la vivienda mediana**; y (6) una **predicción trabajada** de una casa concreta del conjunto de prueba. Cada bloque vuelca su hoja al Excel de resultados.

In [ ]:
# Regla k*MAE: recuento REAL de candidatas a revision en el conjunto de prueba (seed 42).
# Una tasacion es "candidata" si |precio real - precio predicho| > k*MAE del modelo
# multiple. El k operable se fija con la capacidad del equipo: mas k = umbral mas alto =
# menos candidatas (se revisan solo las discrepancias grandes).
mae_mult_test = float(mul.MAE_test)
err_abs_test = np.abs(yte.to_numpy() - m_mult.predict(Xte))

filas_k = []
for k in (1.0, 1.5, 2.0):
    umbral = k * mae_mult_test
    n_cand = int(np.sum(err_abs_test > umbral))
    filas_k.append({"k": k, "umbral_USD": umbral, "candidatas": n_cand,
                    "pct_del_test": 100 * n_cand / len(yte), "n_test": int(len(yte))})
candidatas_k_df = pd.DataFrame(filas_k)

print(f"Conjunto de prueba: {len(yte)} casas  |  MAE multiple = {mae_mult_test:,.0f} USD")
print("Candidatas a revision con |precio real - predicho| > k*MAE:")
for r in filas_k:
    print(f"  k = {r['k']:<4} umbral {r['umbral_USD']:>9,.0f} USD  ->  "
          f"{r['candidatas']:>3} casas ({r['pct_del_test']:.1f} % del test)")

In [ ]:
# Robustez del ahorro por casa: la MISMA tuberia (particion + ajuste + MAE/RMSE) con 5
# semillas de particion. La tabla materializa el rango honesto del titular: el ahorro es
# positivo en las cinco, pero su magnitud depende de la particion (rango, no punto).
filas_sem = []
for s in (0, 1, 7, 42, 123):
    xa_s, xb_s, ya_s, yb_s = train_test_split(d[preds], d["SalePrice"],
                                              test_size=0.2, random_state=s)
    pr_s = LinearRegression().fit(xa_s[["GrLivArea"]], ya_s).predict(xb_s[["GrLivArea"]])
    pr_m = LinearRegression().fit(xa_s, ya_s).predict(xb_s)
    mae_s_s = float(mean_absolute_error(yb_s, pr_s))
    mae_m_s = float(mean_absolute_error(yb_s, pr_m))
    rmse_s_s, rmse_m_s = rmse(yb_s, pr_s), rmse(yb_s, pr_m)
    filas_sem.append({"semilla": s, "MAE_simple_USD": mae_s_s, "MAE_multiple_USD": mae_m_s,
                      "ahorro_MAE_USD_casa": mae_s_s - mae_m_s,
                      "RMSE_simple_USD": rmse_s_s, "RMSE_multiple_USD": rmse_m_s,
                      "reduccion_RMSE_pct": 100 * (rmse_m_s - rmse_s_s) / rmse_s_s})
ahorro_semillas_df = pd.DataFrame(filas_sem)

print("Ahorro de error por casa (MAE simple - MAE multiple) en 5 semillas de particion:")
print(ahorro_semillas_df.round(1).to_string(index=False))
_a = ahorro_semillas_df["ahorro_MAE_USD_casa"]
print(f"\nRango del ahorro: {_a.min():,.0f} a {_a.max():,.0f} USD/casa (positivo en las 5); "
      f"reduccion RMSE: {ahorro_semillas_df['reduccion_RMSE_pct'].min():.0f} % a "
      f"{ahorro_semillas_df['reduccion_RMSE_pct'].max():.0f} %")

In [ ]:
# Asimetria de los residuales del modelo multiple, EN NIVEL (USD) y en log (contraste).
# Convencion de signo: residual = y - yhat. Skew POSITIVO = cola larga donde y > yhat, es
# decir, donde el modelo SUB-predice (infra-valua). El costo del error de la guia (3.5) se
# computa en USD: la asimetria que le corresponde es la de NIVEL, no la del modelo en log.
resid_nivel_in = ols_nivel.resid                       # in-sample, USD (y - yhat)
resid_nivel_te = yte.to_numpy() - m_mult.predict(Xte)  # test seed 42, USD
resid_log_in = ols_log.resid                           # in-sample, log (el -0.94 de B.4)

filas_asi = []
for nombre, e in [("multiple nivel (USD) - in-sample", resid_nivel_in),
                  ("multiple nivel (USD) - test seed 42", resid_nivel_te),
                  ("multiple log - in-sample (contraste)", resid_log_in)]:
    jb_s, jb_p, sk, ku = jarque_bera(np.asarray(e))
    filas_asi.append({"residuales": nombre, "n": int(len(e)), "skew": float(sk),
                      "curtosis": float(ku), "JB": float(jb_s), "JB_p": float(jb_p),
                      "cola_larga": "infra-valuacion (y > yhat)" if sk > 0
                                    else "sobre-valuacion (yhat > y)"})
asimetria_df = pd.DataFrame(filas_asi)
print("Asimetria de residuales (residual = y - yhat; skew > 0 = el modelo SUB-predice la cola):")
print(asimetria_df.round(4).to_string(index=False))

In [ ]:
# Coeficientes ESTANDARIZADOS: que palanca mueve mas el precio por un movimiento COMPARABLE.
# beta_std = beta * sd(x) / sd(y): efecto en desviaciones estandar de y por +1 sd de x.
# Comparar coeficientes en unidades naturales (9.7 %/punto vs 0.028 %/pie2) depende de la
# unidad elegida; +1 sd es el movimiento tipico observable de cada palanca.
sd_lny = float(np.log(d["SalePrice"]).std(ddof=1))
sd_y = float(d["SalePrice"].std(ddof=1))
filas_bs = []
for p in preds:
    sx = float(d[p].std(ddof=1))
    b_l, b_n = float(ols_log.params[p]), float(ols_nivel.params[p])
    filas_bs.append({"predictor": p, "sd_x": sx, "beta_log": b_l,
                     "beta_std_log": b_l * sx / sd_lny,
                     "beta_std_nivel": b_n * sx / sd_y,
                     "efecto_pct_por_1sd": 100 * (np.exp(b_l * sx) - 1)})
betas_std_df = (pd.DataFrame(filas_bs)
                .sort_values("beta_std_log", ascending=False).reset_index(drop=True))
print("Coeficientes estandarizados del modelo multiple (palanca mas potente primero):")
print(betas_std_df.round(4).to_string(index=False))
lider = betas_std_df.iloc[0]
print(f"\nPor movimiento comparable (+1 sd), la palanca mas potente es {lider.predictor} "
      f"(beta_std log {lider.beta_std_log:+.3f}; +1 sd = {lider.sd_x:,.0f} unidades = "
      f"{lider.efecto_pct_por_1sd:+.1f} % del precio).")

In [ ]:
# Palancas en USD sobre la vivienda tipica: mediana (y media) del set limpio, modelo log.
# palanca = precio_base * efecto% / 100. Con esta hoja las cifras 163 000 / ~15 900 /
# ~11 700 / ~6 700 de la guia quedan trazables al Excel (antes solo vivian en el validador).
mediana_precio = float(d["SalePrice"].median())
media_precio = float(d["SalePrice"].mean())
ef_calidad = 100 * (np.exp(ols_log.params["OverallQual"]) - 1)
ef_garaje = 100 * (np.exp(ols_log.params["GarageCars"]) - 1)
ef_premium = 100 * (np.exp(ols_cat.params["premium"]) - 1)

palancas_df = pd.DataFrame([
    {"concepto": "precio_mediano_set_limpio", "efecto_pct": np.nan,
     "sobre_mediana_USD": mediana_precio, "sobre_media_USD": media_precio},
    {"concepto": "palanca_calidad_+1_punto", "efecto_pct": float(ef_calidad),
     "sobre_mediana_USD": mediana_precio * ef_calidad / 100,
     "sobre_media_USD": media_precio * ef_calidad / 100},
    {"concepto": "palanca_garaje_+1_plaza", "efecto_pct": float(ef_garaje),
     "sobre_mediana_USD": mediana_precio * ef_garaje / 100,
     "sobre_media_USD": media_precio * ef_garaje / 100},
    {"concepto": "palanca_premium_dummy", "efecto_pct": float(ef_premium),
     "sobre_mediana_USD": mediana_precio * ef_premium / 100,
     "sobre_media_USD": media_precio * ef_premium / 100},
])
print(f"Vivienda mediana del set limpio: {mediana_precio:,.0f} USD "
      f"(media: {media_precio:,.0f} USD)")
print(palancas_df.round(1).to_string(index=False))

In [ ]:
# Prediccion TRABAJADA de una casa concreta del conjunto de prueba: la de precio real mas
# cercano a la mediana (163 000 USD). Calculo completo: valores de los 5 predictores ->
# yhat en log -> retro-transformacion exp() -> USD -> residual real (real - predicho).
idx_casa = (yte - d["SalePrice"].median()).abs().idxmin()
x_casa = d.loc[idx_casa, preds]
contrib = {p: float(ols_log.params[p]) * float(x_casa[p]) for p in preds}
yhat_log_casa = float(ols_log.params["const"]) + sum(contrib.values())
yhat_usd_casa = float(np.exp(yhat_log_casa))
precio_real_casa = float(d.loc[idx_casa, "SalePrice"])
resid_casa = precio_real_casa - yhat_usd_casa

print(f"Casa Id {int(ames.loc[idx_casa, 'Id'])} del conjunto de prueba (seed 42):")
print(f"  {'intercepto':16s}                                   -> {float(ols_log.params['const']):+9.5f}")
for p in preds:
    print(f"  {p:16s} = {float(x_casa[p]):>7,.0f}  x beta {float(ols_log.params[p]):+.6f} "
          f" -> {contrib[p]:+9.5f}")
print(f"  yhat (log)  = {yhat_log_casa:.5f}")
print(f"  yhat (USD)  = exp({yhat_log_casa:.5f}) = {yhat_usd_casa:,.0f} USD")
print(f"  precio real = {precio_real_casa:,.0f} USD  ->  residual = {resid_casa:+,.0f} USD")
print(f"  |residual| = {abs(resid_casa):,.0f} USD, muy por debajo del MAE "
      f"({mae_mult_test:,.0f} USD): caso tipico, no cereza.")

prediccion_df = pd.DataFrame([{
    "Id": int(ames.loc[idx_casa, "Id"]),
    **{p: float(x_casa[p]) for p in preds},
    "intercepto_log": float(ols_log.params["const"]),
    "yhat_log": yhat_log_casa, "yhat_USD": yhat_usd_casa,
    "precio_real_USD": precio_real_casa, "residual_USD": resid_casa}])

📖 **Lectura (decisión operativa).**

- **Candidatas por `k·MAE`.** Sobre las 292 casas de prueba: **k = 1 → 115 casas (39 %)**, **k = 1.5 → 63 (22 %)**, **k = 2 → 36 (12 %)**. Subir `k` encarece el umbral y acorta la lista; el `k` operable se fija con la **capacidad real** del equipo de revisión, no con el número redondo.
- **Asimetría en USD.** El costo del error se paga en USD, y ahí el skew de los residuales del múltiple es **positivo** (+1.12 in-sample; +0.22 en prueba): la cola larga está del lado de la **infra-valuación** — el modelo subestima el precio de algunas casas de valor elevado. El **−0.94** de la tabla de supuestos es de los residuales del modelo **en log** (otra escala): no se traslada al costo en USD.
- **Palanca más potente.** Por unidad natural la calidad aporta el mayor % por paso (+9.7 %/punto), pero por **movimiento comparable (+1 desviación estándar)** predomina la **superficie** (β_std **+0.356** vs **+0.322** de calidad; +15.1 % vs +13.6 % del precio por +1 sd): «la palanca más potente» depende del criterio de comparación, y el criterio comparable es la sd.
- **Predicción trabajada.** La casa **Id 736** (1 768 pie², calidad 7, 2 plazas, sótano 880 pie², año 1914) obtiene ŷ = 12.024 en log → **exp() ≈ 166 669 USD** frente a un precio real de **163 000 USD**: residual **−3 669 USD** (el modelo la sobre-predice levemente, bien por debajo del MAE de ~24 000). El `exp()` directo entrega la mediana condicional del precio (sin corrección de sesgo); a nivel EPE se lee como la predicción del modelo en USD.

In [ ]:
# --- Exportar TODOS los resultados a resultados/E02_resultados.xlsx ---
simple_adv = pd.DataFrame([
    {"canal": "TV", "b1_por_mil_usd": ols_tv.params["TV"], "R2": ols_tv.rsquared,
     "p_valor": ols_tv.pvalues["TV"], "corr": adv["TV"].corr(adv["sales"]),
     "IC95_low": ic[0], "IC95_high": ic[1]},
    {"canal": "radio", "b1_por_mil_usd": ols_radio.params["radio"], "R2": ols_radio.rsquared,
     "p_valor": ols_radio.pvalues["radio"], "corr": adv["radio"].corr(adv["sales"]),
     "IC95_low": np.nan, "IC95_high": np.nan},
])
coef_nivel = pd.DataFrame({
    "predictor": preds,
    "coef_USD_por_unidad": [ols_nivel.params[p] for p in preds],
    "p_valor": [ols_nivel.pvalues[p] for p in preds]})
coef_log = pd.DataFrame({
    "predictor": preds,
    "efecto_pct_por_unidad": [100 * (np.exp(ols_log.params[p]) - 1) for p in preds],
    "p_valor": [ols_log.pvalues[p] for p in preds]})
categoricas = pd.DataFrame([{
    "variable": "premium (NridgHt/NoRidge/StoneBr)", "R2_sin": ols_log.rsquared,
    "R2_con": ols_cat.rsquared, "coef": ols_cat.params["premium"],
    "efecto_pct": efecto_premium, "p_valor": ols_cat.pvalues["premium"]}])
multicol = pd.DataFrame([
    {"escenario": "GarageCars solo", "coef_GarageCars": coef_solo,
     "corr_GarageCars_GarageArea": corr_garaje},
    {"escenario": "GarageCars + GarageArea (redundante)", "coef_GarageCars": coef_ambas,
     "corr_GarageCars_GarageArea": corr_garaje}])

with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
    # --- 7 hojas de contrato + 2 hojas HC3 (byte-identicas al rework) ---
    simple_adv.to_excel(w, sheet_name="simple_advertising", index=False)
    coef_nivel.to_excel(w, sheet_name="multiple_ames_nivel", index=False)
    coef_log.to_excel(w, sheet_name="multiple_ames_log", index=False)
    categoricas.to_excel(w, sheet_name="categoricas", index=False)
    multicol.to_excel(w, sheet_name="multicolinealidad", index=False)
    corr_pred.round(4).to_excel(w, sheet_name="correlacion_predictores")
    eval_df.to_excel(w, sheet_name="evaluacion", index=False)
    hc3_ames_df.to_excel(w, sheet_name="inferencia_robusta_ames", index=False)
    bp_ames_df.to_excel(w, sheet_name="heterocedasticidad_bp", index=False)
    # --- 2 hojas NUEVAS de incertidumbre/supuestos (no tocan las anteriores) ---
    boot_tv_df.to_excel(w, sheet_name="bootstrap_ic_tv", index=False)
    supuestos_df.to_excel(w, sheet_name="supuestos_diagnostico", index=False)
    # --- 1 hoja NUEVA de valor monetario del -43 % (no toca las anteriores) ---
    valor_error_df.to_excel(w, sheet_name="valor_monetario_error", index=False)
    # --- 6 hojas NUEVAS del arreglo 15/08/2026 (no tocan las anteriores) ---
    candidatas_k_df.to_excel(w, sheet_name="candidatas_por_k", index=False)
    ahorro_semillas_df.to_excel(w, sheet_name="ahorro_semillas", index=False)
    asimetria_df.to_excel(w, sheet_name="asimetria_usd", index=False)
    betas_std_df.to_excel(w, sheet_name="betas_estandarizados", index=False)
    palancas_df.to_excel(w, sheet_name="palancas_mediana", index=False)
    prediccion_df.to_excel(w, sheet_name="prediccion_ejemplo", index=False)

hojas = ["simple_advertising", "multiple_ames_nivel", "multiple_ames_log", "categoricas",
         "multicolinealidad", "correlacion_predictores", "evaluacion",
         "inferencia_robusta_ames", "heterocedasticidad_bp",
         "bootstrap_ic_tv", "supuestos_diagnostico", "valor_monetario_error",
         "candidatas_por_k", "ahorro_semillas", "asimetria_usd",
         "betas_estandarizados", "palancas_mediana", "prediccion_ejemplo"]
print("Excel de resultados escrito en:", XLSX)
print(f"Hojas ({len(hojas)}):", hojas)

In [ ]:
# --- Figuras de RESULTADOS: se generan LEYENDO el Excel (convencion del curso) ---
# F1) Efecto (%) de cada predictor en el precio (modelo en log).
cl = pd.read_excel(XLSX, sheet_name="multiple_ames_log").sort_values("efecto_pct_por_unidad")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
bars = ax.barh(cl["predictor"], cl["efecto_pct_por_unidad"], color=UPC_RED)
ax.bar_label(bars, fmt="%+.2f%%", padding=3, fontsize=9)
ax.set_title("Efecto en el precio por unidad de cada predictor (log)")
ax.set_xlabel("% de cambio en el precio por unidad")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_coef_efectos.png"), bbox_inches="tight")
plt.show()

# F2) RMSE de prueba: modelo simple vs multiple (Ames).
ev = pd.read_excel(XLSX, sheet_name="evaluacion")
ev_ames = ev[ev["modelo"].str.startswith("Ames")]
fig, ax = plt.subplots(figsize=(6.0, 3.8))
bars = ax.bar(["simple\n(GrLivArea)", "multiple\n(5 predictores)"], ev_ames["RMSE_test"],
              color=[UPC_GRAY, UPC_RED])
ax.bar_label(bars, labels=[f"{v:,.0f} USD" for v in ev_ames["RMSE_test"]], padding=3)
ax.set_title("Error de prediccion fuera de muestra (RMSE test)")
ax.set_ylabel("RMSE en USD"); ax.set_ylim(0, ev_ames["RMSE_test"].max() * 1.2)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_eval_rmse.png"), bbox_inches="tight")
plt.show()

# F3) Idea de multicolinealidad: el coeficiente de GarageCars se desestabiliza.
mc = pd.read_excel(XLSX, sheet_name="multicolinealidad")
fig, ax = plt.subplots(figsize=(6.2, 3.8))
bars = ax.bar(mc["escenario"], mc["coef_GarageCars"], color=[UPC_RED, UPC_GRAY])
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_title(f"Coef. de GarageCars al anadir una variable redundante\n"
             f"(corr con GarageArea = {mc['corr_GarageCars_GarageArea'].iloc[0]:.2f})")
ax.set_ylabel("coeficiente (log-precio)")
ax.tick_params(axis="x", labelsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_multicolinealidad.png"), bbox_inches="tight")
plt.show()

# F4) Mapa de correlacion entre predictores (leido del Excel).
cp = pd.read_excel(XLSX, sheet_name="correlacion_predictores", index_col=0)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
sns.heatmap(cp, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1, square=True,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlacion entre predictores (Ames)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_correlacion_predictores.png"), bbox_inches="tight")
plt.show()

# F5) Cifra insignia: IC clasico vs IC bootstrap de la pendiente de TV (leido del Excel).
bt = pd.read_excel(XLSX, sheet_name="bootstrap_ic_tv").iloc[0]
fig, ax = plt.subplots(figsize=(6.6, 2.7))
for j, (lo, hi, col) in enumerate([
        (bt["IC95_clasico_low"], bt["IC95_clasico_high"], UPC_GRAY),
        (bt["IC95_boot_low"], bt["IC95_boot_high"], UPC_RED)]):
    ax.plot([lo, hi], [j, j], color=col, linewidth=3, solid_capstyle="round")
    ax.plot([lo, hi], [j, j], "|", color=col, markersize=13, markeredgewidth=2.4)
ax.axvline(bt["punto"], color=UPC_INK, linestyle=":", linewidth=1.2, alpha=0.7,
           label=f"punto OLS = {bt['punto']:.4f}")
ax.set_yticks([0, 1]); ax.set_yticklabels(["IC clasico", f"IC bootstrap\n(B={int(bt['B'])}, seed {int(bt['semilla'])})"])
ax.set_ylim(-0.8, 1.8); ax.set_xlabel("pendiente de TV (unidades por +1000 USD)")
ax.set_title("Cifra insignia: IC clasico vs IC bootstrap (coinciden)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "e02_bootstrap_ic_tv.png"), bbox_inches="tight")
plt.show()
print("Figuras de resultados guardadas en:", FIG)

**Sintesis de negocio.** La regresion convierte una relacion en una **regla accionable y
medible**: cuanto mueve cada palanca al resultado. La **simple** cuantifica un factor; la
**multiple** aisla el efecto **parcial** de varios a la vez y casi duplica el poder explicativo
del precio de vivienda. Las **categorias** entran como dummies (efecto respecto a una base), y
la decision final se toma por el **error fuera de muestra** (RMSE/MAE en USD), no por el R2 de
entrenamiento. Cuando dos variables miden casi lo mismo (multicolinealidad), sobra una.

**Drills (enunciados; se resuelven en `evaluacion/drills.docx`):**
1. Interpretar un coeficiente en **unidades de negocio** (USD o %).
2. Comparar dos modelos por su **RMSE en datos de prueba**.
3. Anadir una **variable categorica** y leer su efecto respecto a la base.

**Para seguir explorando (fuentes de actualidad).** Ver las fuentes de actualidad de la sesión
(pricing hedonico y valoracion automatizada de vivienda; medicion del retorno publicitario).

> Todas las cifras citadas existen en este cuaderno o en `resultados/E02_resultados.xlsx`.
> Reproducibilidad: `data/descargar_datos.py` obtiene los datasets verificados con checksum.